# The Observability Layer — Theory, This Lab, Evaluation, and What Comes Next

A detailed theoretical explanation of LLM observability, what this lab covers, evaluation theory, what it deliberately does not cover, and what could be added in future versions.

---
## Table of Contents
1. [What Is Observability?](#1)
2. [Observability vs Monitoring vs Logging](#2)
3. [The Three Pillars of Observability](#3)
4. [Why LLM Observability Is Different](#4)
5. [OpenTelemetry — The Standard](#5)
6. [OpenInference — LLM-Specific Semantic Conventions](#6)
7. [This Lab's Architecture](#7)
8. [V1 — Tracing Pipeline Deep Dive](#8)
9. [V2 — Metrics & SLO Pipeline Deep Dive](#9)
10. [SLOs, SLAs, and Error Budgets](#10)
11. [Cost Telemetry — Three Tiers](#11)
12. [LLM Evaluation — Theory and Approaches](#12)
13. [This Lab's Evaluation Approach](#13)
14. [What This Lab Does NOT Do (By Design)](#14)
15. [What Could Be Added Next](#15)

---
## 1. What Is Observability? <a id='1'></a>

**Observability** is a property of a system that answers the question: *"Can I understand the internal state of this system by looking at its external outputs?"*

The term comes from **control theory** (Rudolf Kalman, 1960): a system is *observable* if its internal state can be inferred from its outputs. In software engineering, we make systems observable by emitting **structured telemetry** — traces, metrics, and logs — that tell us what happened, when, why, and how long it took.

### The shift from monitoring to observability

| Monitoring (traditional) | Observability (modern) |
|---|---|
| You know what to alert on in advance | You can ask new questions after the fact |
| Pre-defined dashboards | Ad-hoc exploration of high-cardinality data |
| "Is the CPU high?" | "Which user's request caused the CPU spike, and why?" |
| Static thresholds | Statistical analysis (percentiles, distributions) |
| Logs + metrics (separate) | Traces + metrics + logs (correlated) |

---
## 2. Observability vs Monitoring vs Logging <a id='2'></a>

```mermaid
graph TD
    A[Observability] --> B[Traces]
    A --> C[Metrics]
    A --> D[Logs]
    A --> E[Events]
    A --> F[Profiles]
    B --> B1[Where did time go?]
    B --> B2[What called what?]
    C --> C1[How many? How fast?]
    C --> C2[Is it getting worse?]
    D --> D1[What went wrong?]
    D --> D2[What was the error message?]
```

**Monitoring** is the *practice* of watching metrics and alerting on them. **Observability** is the *capability* that makes monitoring effective. You can have monitoring without observability (you'll be blind to new failure modes), but you can't have effective observability without structured telemetry.

### Where logging fits

Logs are **unstructured or semi-structured text events**. They're useful for debugging specific failures but don't scale for aggregate analysis. In this lab, we deliberately do **not** add a logging pipeline (no Loki) — the trace spans and metrics provide sufficient observability for LLM calls without the complexity of a log aggregation system.

---
## 3. The Three Pillars of Observability <a id='3'></a>

| Pillar | Question it answers | This lab's implementation |
|---|---|---|
| **Traces** | Where did time go? What is the parent-> child call chain? | OTel traces -> Langfuse + Phoenix + LangSmith |
| **Metrics** | How many? How fast? Is it getting worse? | OTel metrics -> Prometheus -> Grafana |
| **Logs** | What exactly went wrong? What was the error text? | NOT implemented (by design -- traces carry exception events) |

### Why we skip logs in this lab

LLM observability has a unique property: the **trace span itself carries the full context** — input, output, model, tokens, latency, error. Unlike a microservice where you need logs to understand what happened inside a function, an LLM call's entire state is captured in one span. Adding Loki or Elasticsearch would add infrastructure complexity without meaningful new insight for this scope.

---
## 4. Why LLM Observability Is Different <a id='4'></a>

Traditional software observability focuses on: CPU, memory, request rate, error rate, latency. LLM observability adds entirely new dimensions:

| Traditional dimension | LLM-specific dimension |
|---|---|
| Request latency | Token generation rate (tokens/second) |
| Error rate (HTTP 5xx) | LLM refusal rate, hallucination indicators |
| CPU utilisation | Cost per request (tokens x price) |
| Memory usage | Context window utilisation |
| Throughput (req/s) | Throughput (tokens/s) |
| API status code | Model-specific errors (overloaded, content filter) |
| No prompt privacy concern | Prompts may contain PII/secrets -- must NOT be recorded |

### The prompt privacy problem

In traditional observability, you can safely log request bodies. In LLM observability, the request body IS the prompt — which may contain user PII, API keys, or other sensitive data. This lab:
- **Never records prompt text** in manual spans
- Uses the collector's `attributes/drop_sensitive` processor to strip `gen_ai.prompt`, `gen_ai.completion`, `llm.input_messages`, `llm.output_messages` as a safety net

### The cost problem

Traditional API calls have a fixed cost per request. LLM calls have **variable cost** based on token count, which varies with prompt length and output length. This lab implements three cost tiers:
1. **Provider-reported cost** -- Groq doesn't report $ in usage -> always `None`
2. **Configured estimated cost** -- computed from token counts x configurable per-million-token price
3. **Missing pricing** -- model not in catalog -> `None`, surfaced as "missing pricing", never `$0`

---
## 5. OpenTelemetry — The Standard <a id='5'></a>

[OpenTelemetry (OTel)](https://opentelemetry.io) is a CNCF project that provides a **vendor-neutral** standard for generating, collecting, and exporting telemetry.

### Architecture

```mermaid
graph LR
    subgraph Your Application
    A[OTel SDK] -->|TracerProvider| B[Span creation]
    A -->|MeterProvider| C[Metric creation]
    end
    B --> D[SpanProcessor]
    C --> E[MetricReader]
    D -->|OTLP| F[Collector]
    E -->|OTLP| F
    F -->|OTLP/HTTP| G[Langfuse]
    F -->|OTLP/gRPC| H[Phoenix]
    F -->|OTLP/HTTPS| I[LangSmith]
    F -->|Prometheus| J[Prometheus]
```

### Key OTel concepts used in this lab

| Concept | What it means | Our implementation |
|---|---|---|
| **TracerProvider** | Creates tracers; owns span processors and exporters | `configure_otel()` -- one instance, never recreated |
| **MeterProvider** | Creates meters; owns metric readers and exporters | `configure_metrics()` -- one instance, never recreated |
| **Span** | A unit of work with start time, end time, attributes, status | Manual workflow spans + auto-instrumented Groq spans |
| **Span context** | Contains trace_id + span_id + parent span_id | Propagates automatically via context manager |
| **Span processor** | Processes spans before export (batching, sampling) | `BatchSpanProcessor` with 32-batch, 5s timeout |
| **Metric reader** | Reads and exports metrics periodically | `PeriodicExportingMetricReader` with 2s interval |
| **OTLP** | OpenTelemetry Protocol -- the wire format | HTTP/protobuf on port 4318, gRPC on 4317 |
| **Resource** | Identifies the service that produced telemetry | `service.name=llm-observability-lab`, `service.version=0.2.0` |

### Why one provider, one exporter

The brief explicitly says: *"Avoid installing three competing global tracer providers or instrumenting Groq multiple times."* 

If you call `trace.set_tracer_provider()` twice, the second call is silently ignored (OTel SDK guards against it). But if you call `GroqInstrumentor().instrument()` twice, you get **double instrumentation** — every Groq call produces two spans.

---
## 6. OpenInference — LLM-Specific Semantic Conventions <a id='6'></a>

[OpenInference](https://github.com/Arize-ai/openinference) is an open standard for LLM observability attributes, maintained by Arize (the Phoenix team). It sits on top of OpenTelemetry and adds attributes specific to LLM/GenAI workloads.

### Standard OTel GenAI attributes vs OpenInference attributes

| Standard GenAI (`gen_ai.*`) | OpenInference (`llm.*`) | Who prefers it |
|---|---|---|
| `gen_ai.system` | (implicit in span kind) | Langfuse, LangSmith |
| `gen_ai.operation.name` | `openinference.span.kind` | Both |
| `gen_ai.request.model` | `llm.model_name` | Phoenix prefers `llm.*` |
| `gen_ai.usage.input_tokens` | `llm.token_count.prompt` | Phoenix prefers `llm.*` |
| `gen_ai.usage.output_tokens` | `llm.token_count.completion` | Phoenix prefers `llm.*` |

### This lab's solution — set both

Every manual span sets **both** `gen_ai.*` and `llm.*` attributes with identical values. This ensures all three backends render tokens correctly without double counting.

---
## 7. This Lab's Architecture <a id='7'></a>

```mermaid
graph TB
    subgraph Python Scripts
    S1[01_single_call.py]
    S2[02_multiple_calls.py]
    S3[03_concurrent_calls.py]
    S4[04_failures.py]
    S5[05_batch_dashboard.py]
    S6[06_synthetic_metrics.py]
    S7[07_evaluation.py]
    end

    subgraph observability package
    O1[instrumentation.py - Traces]
    O2[metrics.py - Metrics]
    O3[pricing.py - Cost]
    O4[usage.py - Tokens]
    O5[slos.py - SLOs]
    O6[metadata.py - Attributes]
    end

    S1 & S2 & S3 & S4 & S5 & S6 & S7 --> O1 & O2 & O3 & O4 & O5 & O6
    O1 -->|OTLP/HTTP traces| C[OTel Collector :4318]
    O2 -->|OTLP/HTTP metrics| C
    C -->|traces| L[Langfuse :3000]
    C -->|traces| P[Phoenix :6006]
    C -->|traces| LS[LangSmith SaaS]
    C -->|metrics| PR[Prometheus :9095]
    PR --> G[Grafana :3001]
```

### Components

| Component | Technology | Purpose |
|---|---|---|
| **Groq SDK** | `groq` Python package | Makes LLM API calls |
| **OpenInference** | `openinference-instrumentation-groq` | Auto-instruments Groq calls |
| **OpenTelemetry SDK** | `opentelemetry-api`, `opentelemetry-sdk` | Creates traces and metrics |
| **OTLP exporters** | `opentelemetry-exporter-otlp-proto-http` | Sends telemetry to collector |
| **OTel Collector** | `otel/opentelemetry-collector-contrib` | Routes telemetry to backends |
| **Langfuse** | Self-hosted Docker | Trace storage + dashboards + scores |
| **Phoenix** | Self-hosted Docker | Trace storage + waterfall UI |
| **LangSmith** | SaaS | Cloud trace storage + UI |
| **Prometheus** | Self-hosted Docker | Time-series metrics storage (45-day retention) |
| **Grafana** | Self-hosted Docker | Dashboarding + alerting |

---
## 8. V1 — Tracing Pipeline Deep Dive <a id='8'></a>

### The collector's trace pipeline

```yaml
# Simplified from infrastructure/otel-collector.yaml
service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [attributes/drop_sensitive, batch]
      exporters:
        - otlphttp/langfuse
        - otlp/phoenix
        - otlphttp/langsmith
```

### The attributes/drop_sensitive processor

Deletes attribute keys that might carry secrets before the spans reach any backend: `gen_ai.prompt`, `gen_ai.completion`, `llm.input_messages`, `llm.output_messages`, `authorization`, `x-api-key`.

### Credentials outside the config

All secrets are read from `${env:...}` — the collector container's environment variables, set in `infrastructure/collector.env` (auto-generated by `sync-keys.ps1` from `.env`).

---
## 9. V2 — Metrics & SLO Pipeline Deep Dive <a id='9'></a>

### The collector's metrics pipeline

```yaml
service:
  pipelines:
    metrics:
      receivers: [otlp]
      processors: [batch/metrics]
      exporters:
        - prometheus       # exposes at 0.0.0.0:8888
```

### Metrics created by this lab

| Metric name (in Prometheus) | Type | Labels |
|---|---|---|
| `llm_obs_llm_requests_total` | counter | service, model, workflow, environment, result |
| `llm_obs_llm_tokens_total` | counter | + token_type (input/output) |
| `llm_obs_llm_estimated_cost_usd_USD_total` | counter | service, model, workflow, environment |
| `llm_obs_llm_api_attempts_total` | counter | + outcome, attempt |
| `llm_obs_llm_request_duration_seconds` | histogram | + le (bucket boundary) |

### The histogram bucket at 4 seconds

The SLO requires ">= 95% under 4 seconds". The histogram buckets include `4.0` so Prometheus can compute:

```promql
sum(llm_obs_llm_request_duration_seconds_bucket{le="4"})
/ clamp_min(sum(llm_obs_llm_request_duration_seconds_count), 1)
```

### Labels — bounded by design

No user IDs, trace IDs, or prompt text as labels. High-cardinality labels would explode Prometheus series count. Total series estimate: ~22,500 — well within Prometheus's comfort zone.

---
## 10. SLOs, SLAs, and Error Budgets <a id='10'></a>

### Definitions

| Term | Meaning | Who sets it |
|---|---|---|
| **SLA** (Service Level Agreement) | Contractual obligation between provider and customer | Legal/business team |
| **SLO** (Service Level Objective) | Internal target that the engineering team aims for | Engineering team |
| **SLI** (Service Level Indicator) | The actual measured metric | Observability system |

**Critical rule:** An internal SLO is NOT a provider's SLA.

### This lab's SLOs

| SLO | Target | Window | SLI |
|---|---|---|---|
| Request success | >= 99.5% | 7 days | `success_count / total_count` |
| Latency | >= 95% under 4s | 1 hour | `count(le=4) / count(total)` |

### Error budget

For a 99.5% success SLO over 7 days:

- allowed_errors = total x 0.005
- budget_consumed = actual_errors / allowed_errors
- budget_remaining = 1 - budget_consumed

| Budget consumed | Meaning |
|---|---|
| 0.0 | No errors -- full budget remaining |
| 1.0 | Budget exhausted -- SLO violated |
| >1.0 | Budget overspent |

### Insufficient-data state

When `total_requests < 100`, SLO evaluations return `None` and `insufficient_data=True`. This prevents misleading percentages from tiny sample sizes and suppresses alert storms during cold starts.

---
## 11. Cost Telemetry — Three Tiers <a id='11'></a>

```python
# Tier 1: Provider-reported cost
# Groq's usage payload does NOT include monetary cost -> always None

# Tier 2: Configured estimated cost
cost = estimate_cost(usage, model='allam-2-7b')
# -> input_cost = input_tokens * 0.10 / 1_000_000
# -> output_cost = output_tokens * 0.20 / 1_000_000
# -> source = 'configured'  (NOT 'provider')

# Tier 3: Missing pricing
cost = estimate_cost(usage, model='unknown-model-xyz')
# -> None  (model not in DEFAULT_CATALOG)
```

### Why three tiers?

1. **Provider-reported** is the actual bill — but Groq doesn't report it, so it's `None`.
2. **Configured estimate** is our best approximation — computed from token counts and a price table we maintain. It's an estimate, **never an actual bill**.
3. **Missing pricing** is the honest answer when we don't know — it's `None`, not `$0`.

### Why use ONE price config when comparing dashboards?

If Phoenix uses price config A and Langfuse uses price config B, the cost difference would be caused by the price difference, not by observed usage. This lab uses `DEFAULT_CATALOG` in `pricing.py` as the single source of truth.

---
## 12. LLM Evaluation — Theory and Approaches <a id='12'></a>

LLM evaluation is the process of **scoring LLM outputs** against quality criteria. Unlike traditional software testing where you assert `expected == actual`, LLM outputs are probabilistic and open-ended — you need different evaluation strategies.

### The evaluation spectrum

```mermaid
graph LR
    A[Heuristic Checks] --> B[LLM-as-Judge]
    B --> C[Human Annotation]
    C --> D[Production A/B Testing]
    
    A --> A1[Keyword matching]
    A --> A2[Length checks]
    A --> A3[Regex / format validation]
    A --> A4[Deterministic — no ML]
    
    B --> B1[GPT-4 grades GPT-3.5 output]
    B --> B2[Self-consistency checks]
    B --> B3[RAGAS: faithfulness, relevance]
    B --> B4[DeepEval: GEval, hallucination]
    
    C --> C1[Thumbs up/down]
    C --> C2[1-5 star rating]
    C --> C3[Error categorization]
    
    D --> D1[Compare model A vs B]
    D --> D2[Compare prompt v1 vs v2]
    D --> D3[Online metrics: CTR, retention]
```

### Approach 1 — Heuristic checks (what this lab uses)

**How it works:** deterministic rules that check if the output meets basic criteria.

| Check | What it verifies | Example |
|---|---|---|
| **Completeness** | Is the response non-empty? | `len(answer.strip()) > 0` |
| **Relevance** | Does it contain expected keywords? | `"red" in answer.lower()` |
| **Conciseness** | Is it within a token budget? | `output_tokens <= 256` |
| **Success** | Did the API call succeed? | `error is None` |

**Pros:**
- Zero cost — no extra LLM calls needed
- Deterministic — same input always gives same score
- Fast — runs in microseconds
- Transparent — anyone can read the code and understand the check

**Cons:**
- Can't assess nuance (tone, creativity, factual accuracy)
- Keyword matching misses synonyms and paraphrases
- Doesn't catch hallucinations (confident but wrong answers)
- Not suitable for open-ended creative tasks

**When to use:** simple Q&A, classification, structured output, smoke testing, CI/CD gates.

### Approach 2 — LLM-as-Judge (not used in this lab)

**How it works:** a stronger model (e.g., GPT-4) grades the output of a weaker model.

```
Prompt: "Rate this answer from 1-5 for accuracy. Question: {question}. Answer: {answer}"
Judge model returns: 4 — "Mostly correct but misses one detail"
```

**Frameworks:**
- **RAGAS** — evaluates RAG pipelines (faithfulness, answer relevance, context precision/recall)
- **DeepEval** — LLM evaluation framework (GEval, hallucination, toxicity, bias)
- **LangChain Eval** — LangSmith's built-in evaluators
- **Promptfoo** — CLI-based prompt testing and comparison

**Pros:**
- Can assess nuance, creativity, factual accuracy
- Scales better than human annotation
- Frameworks provide standardized metrics

**Cons:**
- Cost — each evaluation is another LLM call
- Non-deterministic — same input may get different scores across runs
- Judge bias — models prefer their own style (self-preference bias)
- Complexity — need to tune judge prompts, handle edge cases

**When to use:** RAG quality, open-ended generation, factual accuracy, production quality monitoring.

### Approach 3 — Human annotation (not used in this lab)

**How it works:** humans review and score outputs.

**Tools:**
- **Langfuse Annotation Queues** — built-in human review workflow
- **Argilla** — open-source annotation platform
- **Label Studio** — general-purpose data labeling

**Pros:**
- Highest quality ground truth
- Can catch subtle errors that LLM judges miss
- Human judgment for subjective criteria

**Cons:**
- Slow — humans take seconds to minutes per sample
- Expensive — requires paid annotators
- Not scalable for high-volume evaluation
- Inter-annotator disagreement needs reconciliation

**When to use:** golden dataset creation, production incident review, subjective quality assessment.

### Approach 4 — Production A/B testing (not used in this lab)

**How it works:** route real traffic between model versions and compare business metrics.

**Pros:** measures real user impact, not synthetic quality.
**Cons:** requires production traffic, statistical significance takes time, ethical concerns.

### How scores flow into observability platforms

Regardless of the evaluation approach, scores need to be **attached to traces** so they're visible alongside the LLM call that produced them:

```mermaid
graph LR
    A[LLM Call] -->|produces| B[Output]
    B -->|evaluated by| C[Evaluator]
    C -->|score=0.8| D[Span Attribute]
    D -->|flows through| E[OTel Collector]
    E -->|to| F[Langfuse Scores Column]
    E -->|to| G[Phoenix Span Attributes]
    E -->|to| H[LangSmith Run Metadata]
```

**Langfuse-specific score attributes:**
- `langfuse.score.<name>.value` — numeric score (0.0–1.0)
- `langfuse.score.<name>.comment` — text comment
- These appear in Langfuse's Scores column and Scores dashboard

**Generic attributes (all platforms):**
- `eval.<name>` — simple numeric score
- `eval.framework` — which evaluator produced the score
- Visible in Phoenix and LangSmith as span attributes

---
## 13. This Lab's Evaluation Approach <a id='13'></a>

This lab uses **Approach 1 — heuristic checks** (the simplest, most transparent approach).

### Script: `examples/07_evaluation.py`

**5 test cases** with known prompts and expected keywords:

| Case | Prompt | Expected keywords |
|---|---|---|
| `primary_colors` | "List three primary colors." | red, blue, yellow |
| `chemical_symbol_water` | "What is the chemical symbol for water?" | h2o |
| `simple_math` | "What is 2 + 2?" | 4 |
| `planet_name` | "Name a planet in our solar system." | mercury, venus, earth, ... |
| `programming_language` | "Name a programming language." | python, java, javascript, ... |

**4 checks per case:**

| Check | Scoring logic | Score range |
|---|---|---|
| `success` | 1.0 if no error, 0.0 if error | 0 or 1 |
| `completeness` | 1.0 if >2 chars, 0.5 if 1 char, 0.0 if empty | 0, 0.5, or 1 |
| `relevance` | 1.0 if any expected keyword found, 0.0 otherwise | 0 or 1 |
| `conciseness` | 1.0 if output_tokens <= limit, 0.5 if over, 0.0 if none | 0, 0.5, or 1 |

**How scores are logged:**

```python
# Each eval case gets its own span under the workflow parent
with tracer.start_as_current_span(f"eval.{case.name}") as span:
    # Make the Groq call (auto-instrumented by OpenInference)
    resp = client.chat.completions.create(...)
    answer = resp.choices[0].message.content

    # Run heuristic checks
    scores = [
        eval_success(error),
        eval_completeness(answer),
        eval_relevance(answer, case.expected_keywords),
        eval_conciseness(usage.output_tokens),
    ]

    # Attach to span -> flows to all 3 platforms via collector
    for s in scores:
        span.set_attribute(f"langfuse.score.{s.name}.value", s.value)
        span.set_attribute(f"langfuse.score.{s.name}.comment", s.comment)
        span.set_attribute(f"eval.{s.name}", s.value)
```

**Trace structure in Langfuse/Phoenix:**

```
evaluation.workflow                    (parent — workflow span)
  |-- eval.primary_colors              (child — eval case span)
  |     |-- Completions                (child — auto-instrumented Groq call)
  |     |-- scores: success=1.0, relevance=1.0, completeness=1.0, conciseness=1.0
  |-- eval.chemical_symbol_water       (child — eval case span)
  |     |-- Completions                (child — auto-instrumented Groq call)
  |     |-- scores: success=1.0, relevance=1.0, ...
  ... (3 more cases)
```

### Why heuristic checks (not RAGAS/DeepEval) for this lab

| Factor | Heuristic | RAGAS/DeepEval |
|---|---|---|
| Extra dependencies | None | RAGAS needs langchain, openai, datasets... |
| Extra API cost | None | Each eval = another LLM call |
| Determinism | Yes — same input = same score | No — LLM judge can vary |
| Transparency | Anyone can read the code | Need to understand judge prompts |
| Setup time | Zero | Need to install, configure, tune |
| Suitable for this lab? | **Yes** — simple Q&A, controlled prompts | Overkill for 5 test cases |

### Verified results (live run)

| Case | Overall | Success | Relevance | Completeness | Conciseness |
|---|---|---|---|---|---|
| primary_colors | 100% | 100% | 100% | 100% | 100% |
| chemical_symbol_water | 100% | 100% | 100% | 100% | 100% |
| simple_math | 75% | 100% | 100% | 50% | 100% |
| planet_name | 100% | 100% | 100% | 100% | 100% |
| programming_language | 100% | 100% | 100% | 100% | 100% |
| **Average** | **95%** | | | | |

Scores appear in Langfuse's Tracing page under the `evaluation.workflow` trace (11 observations: 1 parent + 5 eval case spans + 5 auto-instrumented Groq `Completions` spans).

---
## 14. What This Lab Does NOT Do (By Design) <a id='14'></a>

| Not included | Why |
|---|---|
| **Tempo / Loki** | Traces already go to 3 backends; logs add complexity without LLM-specific value |
| **Kubernetes** | This is a local lab, not a production deployment |
| **Application frontend** | Not a chatbot or API product — just telemetry generation |
| **Separate SLO microservice** | SLOs computed in Prometheus rules + Python module |
| **Prompt management** | Scope is observing calls, not managing prompts |
| **Agents / RAG** | Scope is raw LLM calls, not multi-step reasoning pipelines |
| **LLM-as-judge evaluation** | Heuristic checks are sufficient for this lab's controlled test cases |
| **Application-level routing** | No A/B testing, fallback, or load balancing |
| **Quota enforcement** | The lab conserves the real Groq quota; it doesn't enforce limits |
| **Production rate limiter** | Controlled telemetry generation, not traffic shaping |
| **Authentication layer** | No auth on any endpoint — local development lab |

---
## 15. What Could Be Added Next <a id='15'></a>

### V3 — RAG & Agent Tracing
- Trace multi-step RAG pipelines (retrieval -> augmentation -> generation)
- Agent decision trees as nested spans (each tool call = child span)
- OpenInference instrumentation for LangChain, LlamaIndex, or custom frameworks

### V4 — Advanced LLM Evaluation
- **LLM-as-judge:** use a stronger model to grade outputs on faithfulness, relevance, toxicity
- **RAGAS integration:** faithfulness, answer relevance, context precision/recall for RAG pipelines
- **DeepEval integration:** GEval, hallucination detection, bias checks
- **Golden datasets:** versioned test sets with expected outputs for regression testing
- **Human annotation:** Langfuse annotation queues for manual review workflows
- **A/B comparison:** run the same prompt through two models, compare scores side by side

### V5 — Production Hardening
- Tempo as a fourth trace backend (for teams already in the Grafana ecosystem)
- Loki for structured log correlation with traces
- Kubernetes manifests + Helm chart for production deployment
- Multi-tenant isolation (per-team projects in each backend)
- Cost alerts when estimated spend exceeds a daily/weekly budget

### V6 — Prompt Management
- Versioned prompt templates stored in Langfuse
- Prompt A/B testing with automatic traffic splitting
- Prompt diff visualisation across versions

### V7 — Provider-Reported Cost
- If Groq adds cost to its usage payload, wire it through as Tier 1
- Compare provider-reported vs configured-estimated cost
- Alert on cost discrepancies

### V8 — Multi-Model Routing
- Route requests between models based on cost/latency/quality SLOs
- Observe routing decisions as spans ("why did we pick model X?")
- Cost optimization: automatically choose the cheapest model that meets the latency SLO

---
## Summary — The Mental Model

```
+-------------------------------------------------------------+
|                    YOUR PYTHON CODE                          |
|  scripts that make Groq calls, record metrics,              |
|  and evaluate outputs with heuristic checks                  |
+--------------------------+----------------------------------+
                           |
          +----------------+----------------+
          |  observability package          |
          |  - Traces (OTel SDK)            |
          |  - Metrics (OTel SDK)           |
          |  - Cost (pricing.py)            |
          |  - SLOs (slos.py)               |
          |  - Evaluation (07_evaluation.py)|
          +----------------+----------------+
                           | OTLP/HTTP (port 4318)
          +----------------+----------------+
          |   OTel Collector                |
          |   traces -> 3 backends           |
          |   metrics -> Prometheus          |
          +--+--------+--------+--+----------+
             |        |        |  |
     +-------+  +-----+  +----+  +--------+
     v          v        v                v
  Langfuse   Phoenix  LangSmith     Prometheus
  :3000      :6006    (SaaS)        :9095
     |          |        |                |
     v          v        v                v
  Traces     Traces   Traces          Grafana
  + Scores   + I/O    + Runs          :3001
  + Cost     + Tokens + Usage         + SLOs
  + Latency  + Cost   + Cost          + Alerts
  + Usage    + Eval   + Eval          + Eval
```

**One pipeline, five backends, zero prompt text recorded, bounded labels,
three-tier cost, two SLOs with error budgets, heuristic evaluation with
scores logged to all platforms, insufficient-data protection.**